# LLM Tool Calling — First Principles

## Q1. An LLM can only generate tokens. How can it call tools?

At its core, an LLM is a **next-token predictor**. It does not execute functions. It only generates tokens.

Suppose an LLM generates:

```python
calculate(2, 3)
```

How do we distinguish between:

- **Documentation:** The LLM is showing an example of `calculate(2, 3)`.
- **Tool Calling:** The LLM actually wants to execute `calculate(2, 3)`.

**Solution: Introduce a structured protocol.**

For example, we could design our own format:

Normal response:

```text
<answer>
You can call calculate(2, 3).
</answer>
```

Tool call:

```text
<tool_call>
{"name": "calculate", "arguments": {"a": 2, "b": 3}}
</tool_call>
```

An external program (the *harness*) parses the model's output and distinguishes normal text from tool calls.

The format could use special tokens, structured JSON, or something more sophisticated like OpenAI's **Harmony**, which separates message roles, channels, recipients, bodies, and control-flow endings.

The specific format is not the fundamental idea.

**Key insight: Tool calling is structured token generation + protocol interpretation.**

Without an external program to interpret and execute the request, the LLM still only produces tokens.

---

## Q2. How does an LLM learn when and which tools to call?

An LLM's tool-calling behavior comes from two main sources:

### 1. Training — Knowledge stored in model weights

| Source | What the model learns |
|---|---|
| Pretraining | Knowledge of programming languages, APIs, functions, and tool names |
| Supervised Fine-Tuning (SFT) | When to generate tool calls, which tools to choose, how to format arguments, and how to use tool results |
| Reinforcement Learning (RL) | Tool-use strategies that improve task success through feedback |

For example, repeated training on `bash` tool-use trajectories may make a model particularly familiar with using `bash`.

### 2. Runtime — Knowledge provided through context

| Source | Example |
|---|---|
| Provider context | Tool-related instructions or definitions supplied by the model provider (ex. OpenAI, Anthropic) |
| System / Developer prompt | "Use Python for calculations." |
| User prompt | "Please use the calculator." |
| API `tools` parameter | A new tool's name, description, and parameter schema |

These sources influence the same next-token probability distribution:

$$
P_\theta(\text{next token}\mid\text{context})
$$

The model does not search these sources one by one. Its existing weights and current context jointly determine its output probabilities.

### Hypothesis: Training-Familiar vs. Unseen Tools

We can distinguish:

- **Training-familiar tools:** Tools or interfaces the model has repeatedly encountered during training, such as common shell or Python tool-use patterns.
- **Unseen tools:** Custom tools introduced at runtime, such as a newly created `banana()` function.

**Hypothesis:** All else equal, an LLM may use training-familiar tools more reliably than completely unfamiliar tools.

However, familiarity does not guarantee better performance. A simple, well-documented custom tool may be easier to use than a powerful but complicated familiar tool.

This is an empirical question worth testing with evals.

Crucially, training familiarity does not mean that a tool is physically built into the model.

---

## Q3. What happens when we pass `tools=TOOLS` to an LLM API?

Conceptually, the tool definitions can serve **two different purposes**:

1. Provide knowledge to the LLM through its context.
2. Constrain its generated output using the tool's schema.

### A. Tool Definitions → Model Context

Suppose we pass:

```python
TOOLS = [{
    "name": "multiply",
    "description": "Multiply two integers",
    "parameters": {
        "a": "integer",
        "b": "integer"
    }
}]
```

This is a simplified tool definition, not a complete API schema.

The inference system may transform it into a model-readable representation, conceptually similar to:

```text
Available tools:

multiply(a: integer, b: integer)
Description: Multiply two integers.
```

The process is roughly:

$$
\text{Python Dict}
\xrightarrow{\text{Chat Template}}
\text{Text / Structured Context}
\xrightarrow{\text{Tokenizer}}
\text{Token IDs}
$$

The LLM reads these tokens just like other contextual information.

This explains how a model can use tools it has never encountered during training: **it learns about them from their runtime descriptions.**

The exact serialization varies across models and providers.

### B. JSON Schema → Constrained Decoding

**Problem:** Even if an LLM understands how to call a tool, it might generate malformed JSON or arguments that violate the required schema.

**Solution: Schema-Constrained Decoding.**

Recall that an LLM produces logits for possible next tokens.

Before sampling, the decoder can eliminate tokens that would violate the grammar:

$$
z_i'=
\begin{cases}
z_i & \text{if token is allowed}\\
-\infty & \text{otherwise}
\end{cases}
$$

Then:

$$
P'=\operatorname{softmax}(z')
$$

Invalid token choices receive zero probability.

A grammar engine tracks the partially generated output and updates which tokens are allowed at each step.

**Key insight: Instead of teaching the LLM to never make formatting mistakes, we can make invalid formatting impossible during normal constrained generation.**

Modern models are already good at generating structured outputs. Constrained decoding adds a deterministic guarantee rather than relying only on the model's learned reliability.

For a supported schema, a correctly implemented strict decoder can guarantee schema-valid output **when generation completes successfully**. Refusals, truncation, unsupported schemas, and API failures must still be handled.

This mechanism does not require changing the Transformer architecture.

---

## Final Mental Model

$$
\boxed{
\text{Tool Calling}
=
\text{Learned Structured Generation}
+
\text{Protocol Interpretation}
}
$$

And `tools=TOOLS` can influence the generation process through two complementary paths:

```text
                  tools=TOOLS
                      |
             +--------+--------+
             |                 |
        Chat Template     Schema Compiler
             |                 |
        Model Context     Grammar Constraints
             |                 |
             v                 v
            LLM ----------> Decoder
                                |
                                v
                        Structured Tool Call
                                |
                                v
                             Parser
```

**Three things to remember:**

1. **Training teaches the model how to use tools.** Runtime context introduces the tools available for the current task.
2. **Tool calling is still next-token prediction.** Structured protocols distinguish action requests from ordinary text.
3. **Constrained decoding handles formatting reliability.** It can enforce supported schemas during successful generation, while the model determines what to generate within those constraints.

## Q4. Who is responsible for executing tool calls and returning the results?

Once the LLM generates a tool call, the API exposes it as structured data rather than ordinary assistant text.

For example:

```json
{
  "type": "function_call",
  "name": "multiply",
  "arguments": "{\"a\":2,\"b\":3}",
  "call_id": "call_123"
}
```

For **client-side tools**, the application developer is responsible for connecting the LLM to the environment.

The complete interaction loop is:

1. **LLM API:** The model generates a structured tool call.
2. **Application:** Read the tool name, parse and validate the arguments.
3. **Application:** Execute the corresponding function in the environment.
4. **Application:** Return the result, associated with the original `call_id`.
5. **LLM API:** The model receives the tool result as context and continues generating.


For example, OpenAI's Responses API accepts tool results in this form:

```json
{
  "type": "function_call_output",
  "call_id": "call_123",
  "output": "6"
}
```

The call ID connects the tool result to the original call, particularly important when multiple tools are called.

The model may then produce another tool call, so the application repeats this process until the model finishes its response.

**Important distinctions:**

- API / SDK: Exposes tool calls through a structured protocol, but argument parsing may still be required.
- Constrained Decoder: Can guarantee valid arguments for supported schemas when strict generation completes successfully.
- Application / Harness: Controls execution, validates inputs, handles errors, and returns tool results.
- Provider-Hosted Tools: May be executed automatically by the model provider.

**Key insight: An LLM doesn't execute tools. It generates requests for external actions. Tool calling is the feedback loop connecting those requests to real-world execution.**

Once this mental model is understood, different LLM tool-calling APIs mostly become variations in schema definitions, response formats, and orchestration details.

## Q5. How does an LLM Agent Loop work?

**Core idea: An Agent is an LLM inside a loop that repeatedly calls tools, receives results, and continues generating.**

### 1. The simplest example

User asks:

> What is 2 × 3?

**Step 1 — LLM generates a tool call**

```json
{
  "name": "multiply",
  "arguments": {"a": 2, "b": 3},
  "id": "call_123"
}
```

**Step 2 — Our Python program executes the function**

```python
result = multiply(2, 3)  # 6
```

**Step 3 — Append the tool call and its result to the conversation**

```python
messages = [
    {
        "role": "user",
        "content": "What is 2 × 3?"
    },
    {
        "role": "assistant",
        "content": "I'll calculate that.",
        "tool_calls": [
            {
                "id": "call_123",
                "type": "function",
                "function": {
                    "name": "multiply",
                    "arguments": '{"a": 2, "b": 3}'
                }
            }
        ]
    },
    {
        "role": "tool",
        "tool_call_id": "call_123",
        "content": "6"
    },
    {
        "role": "assistant",
        "content": "2 × 3 = 6."
    }
]
```


This is a simplified illustration, not literal API message syntax.

**Step 4 — Call the LLM again with the updated messages**

The LLM now sees its previous tool call and the result `6`.

It can either generate the final answer or request another tool call.

### 2. The Agent Loop

```python
while True:
    response = llm(messages)

    if not response.tool_calls:
        break

    messages.append(response.assistant_message)

    for call in response.tool_calls:
        result = execute_tool(call)
        messages.append(tool_result(call.id, result))
```

This is conceptual pseudocode; the exact SDK syntax varies.

The loop runs without further user input until the model stops requesting tools (or another stopping condition is reached).

### 3. What exactly happens to the conversation context?

The important detail is that **the LLM does not resume an unfinished sentence**.

Instead, each API call starts a new generation using the updated conversation history:

```text
User: What is 2 × 3?

Assistant → multiply(2, 3) [call_123]

Tool [call_123]: 6

Assistant: 2 × 3 = 6.
```

The application maintains `messages`. The SDK sends them to the API, and the inference system serializes them into the model's context.

The `call_id` links each tool result to its original request, especially when multiple tools are called.

### 4. The mathematical view

Each iteration updates the context:

$$
C_{t+1}=C_t+\text{Tool Call}+\text{Tool Result}
$$

Then the LLM generates again:

$$
\operatorname{LLM}(C_{t+1})
\rightarrow
\begin{cases}
\text{Another Tool Call}\\
\text{Final Answer}
\end{cases}
$$

**Key insight: An LLM Agent is fundamentally a next-token predictor connected to an environment through a feedback loop.**

The LLM decides what action to request. The application executes it, returns the observation, and asks the LLM what to do next.